# 01 · Infrastruttura: ADR, DPS, due IoT Hub e PKI

Un **namespace ADR**, un **DPS**, **due IoT Hub in uksouth**, la **PKI** (Root CA, Issuing CA, policy).
I nomi seguono la convenzione Microsoft (Cloud Adoption Framework):
`<sigla>-<workload>-<ambiente>-<regione>-<istanza>`; i comandi sono quelli veri di Azure CLI, con i valori letti da `_scenario.psd1`.

```text
                 Namespace ADR  adrns-iot-dev-uks  (uksouth)
                 Root CA -> Issuing CA -> policy
                            |
   DPS provs-iot26-dev-uks-001 (uksouth) ---- link ADR
    |-- gruppo "robots" --> IoT Hub iot-iot26-dev-uks-001 (uksouth)       robot-01, robot-02
    '-- gruppo "trucks" --> IoT Hub iot-iot26-dev-uks-002 (uksouth)       truck-01, truck-02
```

**Come si usa.** Esegui le celle in ordine, una alla volta: ogni cella è un passo, e il testo sopra spiega perché. Cambia `$instance` nella prima cella se un nome
globale (Hub, DPS) è già preso. Il notebook 02a continua da qui.

**Vincolo: ADR non supporta il link tra regioni diverse.** Il comando `az iot adr ns link hub add` per un Hub in centralus, con il namespace in uksouth, ha risposto `Cross-region linking is not supported. Namespace region is 'uksouth' and IoT Hub region is 'centralus'`. Per questo namespace, DPS e **entrambi gli Hub sono in uksouth**; centralus, pur essendo nella lista delle regioni in preview, non si può usare per un Hub collegato a questo namespace.


In [ ]:
# Carica la libreria e legge lo scenario (notebooks\_scenario.psd1): da qui in poi si usa $cfg
. ([scriptblock]::Create([IO.File]::ReadAllText((Get-ChildItem . , .\notebooks , .. -Filter _start.ps1 -ErrorAction SilentlyContinue | Select-Object -First 1).FullName)))

Scenario: rg-iot26-dev-uks-001 | namespace ADR adrns-iot-dev-uks | output in D:\Dev\2026\IoTHub\WPC2026\artifacts

## Controlli (sola lettura)
Non crea niente. Serve il ruolo **Owner** nella subscription (le assegnazioni di ruolo dei link) e l'estensione preview `azure-iot` 1.0.0b2.

In [ ]:
az account show --query "{subscription:name, id:id, user:user.name}" -o table

$ext = az extension show --name azure-iot --query version -o tsv 2>$null
if ($ext -eq '1.0.0b2') { "azure-iot $ext pronta." }
else { Write-Warning "azure-iot e' '$ext'. Installa con:  az extension add --name azure-iot --version 1.0.0b2 --allow-preview true --upgrade --yes" }

foreach ($ns in 'Microsoft.Devices', 'Microsoft.DeviceRegistry') {
    "{0,-28} {1}" -f $ns, (az provider show --namespace $ns --query registrationState -o tsv 2>$null)
}

Subscription      User

----------------  ------------------------

IoT Subscription  marco.parenzan@1nn0va.it

azure-iot 1.0.0b2 pronta.

Microsoft.Devices            Registered

Microsoft.DeviceRegistry     Registered

In [ ]:
# L'integrazione ADR + IoT Hub e' in public preview solo in alcune regioni (Microsoft Learn: "What's new in Azure IoT Hub").
$preview = 'australiaeast', 'japaneast', 'centralus', 'uksouth'
$adrRegions = @($adr.Region, $dps.Region) + @($hubs | ForEach-Object { $_.Region })
foreach ($r in ($adrRegions | Select-Object -Unique)) {
    if ($r -in $preview) { "$r : in preview" } else { Write-Warning "$r NON e' tra le regioni in preview per ADR + IoT Hub ($($preview -join ', ')): il link di un Hub qui potrebbe essere rifiutato." }
}
$other = @($hubs | Where-Object { $_.Region -ne $adr.Region })
if ($other.Count -gt 0) { Write-Warning "$($other.Count) Hub sono in una regione diversa dal namespace ($($adr.Region)): ADR rifiuta il link ('Cross-region linking is not supported'). Metti tutti gli Hub in $($adr.Region)." }

""
"Piano:"
"  Resource group : $rg ($rgRegion)"
"  ADR            : $($adr.Name) ($($adr.Region))"
"  DPS            : $($dps.Name) ($($dps.Region))"
foreach ($h in $hubs) {
    $mine = ($devices | Where-Object { $_.Group -eq $h.Group } | ForEach-Object { $_.Id }) -join ', '
    "  IoT Hub        : {0} ({1}) <- gruppo {2}: {3}" -f $h.Name, $h.Region, $h.Group, $mine
}
"  PKI ADR        : $($pki.Root) -> $($pki.Issuing) -> policy $($pki.Policy) ($($pki.ValidityDays) giorni)"
"  File generati  : $out"

uksouth : in preview

Piano:

  Resource group : rg-iot26-dev-uks-001 (uksouth)

  ADR            : adrns-iot-dev-uks (uksouth)

  DPS            : provs-iot26-dev-uks-001 (uksouth)

  IoT Hub        : iot-iot26-dev-uks-001 (uksouth) <- gruppo robots: robot-01, robot-02, device-da-aggiornare

  IoT Hub        : iot-iot26-dev-uks-002 (uksouth) <- gruppo trucks: truck-01, truck-02, device-da-revocare

  PKI ADR        : root-iot26-dev -> ica-iot26-dev -> policy cp-iot26-dev (30 giorni)

  File generati  : D:\Dev\2026\IoTHub\WPC2026\artifacts

## 1 · Resource group e namespace ADR
Un contenitore per tutta la demo: a fine sessione basta eliminare lui. Il **namespace** è il confine logico e di sicurezza della
flotta; l'**identità di sistema** gli permette di parlare con Hub e DPS **senza credenziali salvate**.

In [ ]:
az provider register --namespace Microsoft.Devices --wait
az provider register --namespace Microsoft.DeviceRegistry --wait

az group create --name $rg --location $rgRegion --tags workload=$workload environment=$stage -o table
Assert-Az 'Creazione del resource group'

Location    Name

----------  --------------------

uksouth     rg-iot26-dev-uks-001

In [ ]:
$subscriptionId = az account show --query id -o tsv
$nsUrl = "https://management.azure.com/subscriptions/$subscriptionId/resourceGroups/$rg/providers/Microsoft.DeviceRegistry/namespaces/$($adr.Name)?api-version=2026-11-01"

# Un secondo PUT su un namespace che ha gia' dei link puo' portarlo in stato Failed: se esiste, non lo ricreo.
$exists = az rest --method get --url $nsUrl --query name -o tsv 2>$null
if ($exists) { "Il namespace $($adr.Name) esiste gia': salto la creazione." }
else {
    $body = @{
        location   = $adr.Region
        tags       = @{ workload = $workload; environment = $stage }
        identity   = @{ type = 'SystemAssigned' }
        properties = @{ outboundIdentity = @{ type = 'SystemAssigned' } }
    } | ConvertTo-Json -Depth 10 -Compress
    $bodyFile = Join-Path ([IO.Path]::GetTempPath()) "$($adr.Name)-namespace.json"
    [IO.File]::WriteAllText($bodyFile, $body, [Text.UTF8Encoding]::new($false))

    az rest --method put --url $nsUrl --headers 'Content-Type=application/json' --body "@$bodyFile" -o none
    Assert-Az 'Creazione del namespace ADR'
    Wait-AdrNamespace -Url $nsUrl
    Remove-Item $bodyFile -Force
}

az rest --method get --url $nsUrl --query '{nome:name, regione:location, stato:properties.provisioningState, identita:identity.type}' -o table

Il namespace adrns-iot-dev-uks esiste gia': salto la creazione.

Nome               Regione    Stato      Identita

-----------------  ---------  ---------  --------------

adrns-iot-dev-uks  uksouth    Succeeded  SystemAssigned

## 2 · DPS e link ad ADR
Il **Device Provisioning Service** assegna i device agli Hub. Si crea **prima** del DPS e del suo link, poi gli Hub: il link di un Hub
richiede un DPS già collegato. Un namespace accetta **un solo DPS**, e il link è **irreversibile**.

In [ ]:
$dpsEsiste = az iot dps show --name $dps.Name --resource-group $rg --query name -o tsv --only-show-errors 2>$null
if ($dpsEsiste) { "Il DPS $($dps.Name) esiste gia': salto la creazione (az iot dps create risponde 400307 se il nome c'e' gia')." }
else {
    az iot dps create --name $dps.Name --resource-group $rg --location $dps.Region --system-assigned-mi --tags workload=$workload environment=$stage -o none
    Assert-Az 'Creazione del DPS'
}
$dpsId = az iot dps show --name $dps.Name --resource-group $rg --query id -o tsv --only-show-errors

$dpsLinks = @(az iot adr ns link dps list --namespace $adr.Name --resource-group $rg --query '[].name' -o tsv --only-show-errors)
if ($dpsLinks -contains $dps.Link) { "Il link $($dps.Link) esiste gia': salto (i link ADR sono irreversibili)." }
else {
    az iot adr ns link dps add --name $dps.Link --namespace $adr.Name --resource-group $rg --dps-id $dpsId --system-assigned-mi
    Assert-Az 'Link DPS -> ADR'
}

Il DPS provs-iot26-dev-uks-001 esiste gia': salto la creazione (az iot dps create risponde 400307 se il nome c'e' gia').

Il link adrlink-provs-iot26-dev-uks-001 esiste gia': salto (i link ADR sono irreversibili).

## 3 · Gli Hub
Due IoT Hub **Standard** (S1 o superiore: la SKU F1 non è supportata), **entrambi in uksouth**, la regione del namespace: **ADR non supporta il link tra regioni diverse** (vedi la nota in cima). Il tag `provisioning=adr` documenta che flusso usare.

In [ ]:
$preview = 'australiaeast', 'japaneast', 'centralus', 'uksouth'
foreach ($h in $hubs) {
    "--- $($h.Name) in $($h.Region)"
    $hubEsiste = az iot hub show --name $h.Name --resource-group $rg --query name -o tsv --only-show-errors 2>$null
    if ($hubEsiste) { "L'Hub $($h.Name) esiste gia': salto la creazione." }
    else {
        az iot hub create --name $h.Name --resource-group $rg --location $h.Region --sku $h.Sku --system-assigned-mi --tags workload=$workload environment=$stage provisioning=adr -o none
        Assert-Az "Creazione dell'Hub $($h.Name)"
    }
    $hubId = az iot hub show --name $h.Name --resource-group $rg --query id -o tsv --only-show-errors

    # Il link ADR e' irreversibile e non si ripete: se esiste gia' Succeeded si salta; se esiste ma e' Failed si riprova con 'update', non con 'add'.
    $linkState = az iot adr ns link hub list --namespace $adr.Name --resource-group $rg --query "[?name=='$($h.Link)'].linkingState | [0]" -o tsv --only-show-errors
    if ($linkState -eq 'Succeeded') { "Il link $($h.Link) e' gia' Succeeded: salto."; continue }
    if ($linkState) { az iot adr ns link hub update --name $h.Link --namespace $adr.Name --resource-group $rg }
    else { az iot adr ns link hub add --name $h.Link --namespace $adr.Name --resource-group $rg --hub-id $hubId --system-assigned-mi }
    if ($LASTEXITCODE -ne 0) {
        throw "Link $($h.Name) -> ADR rifiutato. Regione dell'Hub: $($h.Region), regione del namespace: $($adr.Region). ADR NON supporta il link tra regioni diverse: metti l'Hub in $($adr.Region) in `$hubs. Regioni in preview per ADR + IoT Hub: $($preview -join ', '). Un link rimasto Failed si riprova rieseguendo questa cella (usa 'link hub update')."
    }
}

--- iot-iot26-dev-uks-001 in uksouth

L'Hub iot-iot26-dev-uks-001 esiste gia': salto la creazione.

Il link adrlink-iot-iot26-dev-uks-001 e' gia' Succeeded: salto.

--- iot-iot26-dev-uks-002 in uksouth

L'Hub iot-iot26-dev-uks-002 esiste gia': salto la creazione.

Il link adrlink-iot-iot26-dev-uks-002 e' gia' Succeeded: salto.

In [ ]:
# I link si completano in modo asincrono: aspettiamo e guardiamo lo stato.
az iot adr ns link wait --namespace $adr.Name --resource-group $rg
az iot adr ns link dps list --namespace $adr.Name --resource-group $rg -o table
az iot adr ns link hub list --namespace $adr.Name --resource-group $rg -o table

## 4 · La PKI nel namespace
La **fiducia come risorsa**: Root CA → Issuing CA → certificate policy, tre risorse del namespace con tre ruoli.

- **Root CA**: la radice della fiducia. **Non emette certificati ai device**: firma soltanto la Issuing CA.
- **Issuing CA** (tipo ICA, PKI gestita da Microsoft): è quella che **emette** i certificati dei device. La sua firma è garantita dalla Root.
- **Certificate policy**: è legata alla Issuing CA e fissa le regole dei certificati che emette. Qui la validità: 30 giorni.

Le chiavi stanno in un HSM gestito da Microsoft: non c'è nessun file da proteggere. Non è la CA di bootstrap del notebook 02a, che serve a presentare il device a DPS.

In [ ]:
az iot adr ns ca create --name $pki.Root --namespace $adr.Name --resource-group $rg --location $adr.Region --type Root
Assert-Az 'Creazione della Root CA'

{

  "id": "/subscriptions/50a6222f-9144-411d-9ea9-c18d562275f3/resourceGroups/rg-iot26-dev-uks-001/providers/Microsoft.DeviceRegistry/namespaces/adrns-iot-dev-uks/certificateAuthorities/root-iot26-dev",

  "location": "uksouth",

  "name": "root-iot26-dev",

  "properties": {

    "certificateAuthorityType": "Root",

... (14 outputs left out)

    "lastModifiedByType": "User"

  },

  "type": "Microsoft.DeviceRegistry/namespaces/certificateAuthorities"

}

In [ ]:
az iot adr ns ca create --name $pki.Issuing --namespace $adr.Name --resource-group $rg --location $adr.Region --type ICA --issuer-type Microsoft --issuer-ca-name $pki.Root
Assert-Az 'Creazione della Issuing CA'

{

  "id": "/subscriptions/50a6222f-9144-411d-9ea9-c18d562275f3/resourceGroups/rg-iot26-dev-uks-001/providers/Microsoft.DeviceRegistry/namespaces/adrns-iot-dev-uks/certificateAuthorities/ica-iot26-dev",

  "location": "uksouth",

  "name": "ica-iot26-dev",

  "properties": {

    "certificateAuthorityType": "ICA",

... (18 outputs left out)

    "lastModifiedByType": "User"

  },

  "type": "Microsoft.DeviceRegistry/namespaces/certificateAuthorities"

}

In [ ]:
az iot adr ns ca policy create --name $pki.Policy --ca-name $pki.Issuing --namespace $adr.Name --resource-group $rg --location $adr.Region --validity-days $pki.ValidityDays
Assert-Az 'Creazione della certificate policy'

{

  "id": "/subscriptions/50a6222f-9144-411d-9ea9-c18d562275f3/resourceGroups/rg-iot26-dev-uks-001/providers/Microsoft.DeviceRegistry/namespaces/adrns-iot-dev-uks/certificateAuthorities/ica-iot26-dev/certificatePolicies/cp-iot26-dev",

  "name": "cp-iot26-dev",

  "properties": {

    "certificate": {

      "validityPeriodInDays": 30

... (11 outputs left out)

    "lastModifiedByType": "User"

  },

  "type": "Microsoft.DeviceRegistry/namespaces/certificateAuthorities/certificatePolicies"

}

In [ ]:
az iot adr ns ca show --name $pki.Root --namespace $adr.Name --resource-group $rg --query '{nome:name, tipo:properties.certificateAuthorityType}' -o table
az iot adr ns ca show --name $pki.Issuing --namespace $adr.Name --resource-group $rg --query '{nome:name, tipo:properties.certificateAuthorityType}' -o table
az iot adr ns ca policy show --name $pki.Policy --ca-name $pki.Issuing --namespace $adr.Name --resource-group $rg

Nome            Tipo

--------------  ------

root-iot26-dev  Root

Nome           Tipo

-------------  ------

ica-iot26-dev  ICA

... (17 outputs left out)

    "lastModifiedByType": "User"

  },

  "type": "Microsoft.DeviceRegistry/namespaces/certificateAuthorities/certificatePolicies"

}

**Fatto.** L'infrastruttura c'è: namespace ADR, DPS, due Hub collegati e la PKI. Si continua con il notebook **02a · La CA di bootstrap su DPS**.